In [8]:
using LinearAlgebra
using Random
using Printf

n = 1000
k = 0.01

function generateMatrix(n, k)
    A = Float64.(rand(-100:100, n, n)) ./ 100.0
    for i in 1:n
        A[i, i] = 0.0
        rowSum = sum(abs.(A[i, :]))
        for j in 1:n
            A[i, j] = A[i, j] / rowSum
        end
        signValue = rand(Bool) ? 1.0 : -1.0
        A[i, i] = signValue * (1.0 + k)
    end
    return A
end

function isDominant(A, row, column)
    current = abs(A[row, column])
    rowSum = 0.0
    for j in 1:size(A, 2)
        if j != column
            rowSum += abs(A[row, j])
        end
    end
    return current > rowSum
end

function gaussSolve(A, b)
    n = size(A, 1)
    M = copy(A)
    right = copy(b)
    for i in 1:n-1
        for row in i+1:n
            factor = M[row, i] / M[i, i]
            M[row, i] = 0.0
            for column in i+1:n
                M[row, column] -= factor * M[i, column]
            end
            right[row] -= factor * right[i]
        end
    end
    x = zeros(Float64, n)
    for i in n:-1:1
        value = right[i]
        for j in i+1:n
            value -= M[i, j] * x[j]
        end
        x[i] = value / M[i, i]
    end
    return x
end

function reorderRows(A, b)
    n = size(A, 1)
    M = copy(A)
    right = copy(b)
    for column in 1:n
        foundRow = column
        for row in column:n
            if isDominant(M, row, column)
                foundRow = row
                break
            end
        end
        if foundRow != column
            temp = copy(M[column, :])
            M[column, :] = M[foundRow, :]
            M[foundRow, :] = temp
            right[column], right[foundRow] = right[foundRow], right[column]
        end
    end
    return M, right
end

function reorderColumns(A)
    n = size(A, 1)
    M = copy(A)
    order = collect(1:n)
    for row in 1:n
        foundColumn = row
        for column in row:n
            if isDominant(M, row, column)
                foundColumn = column
                break
            end
        end
        if foundColumn != row
            temp = copy(M[:, row])
            M[:, row] = M[:, foundColumn]
            M[:, foundColumn] = temp
            order[row], order[foundColumn] = order[foundColumn], order[row]
        end
    end
    return M, order
end

function reorderCombined(A, b)
    n = size(A, 1)
    M = copy(A)
    right = copy(b)
    order = collect(1:n)
    for i in 1:n
        foundRow = i
        foundColumn = i
        found = false
        for row in i:n
            for column in i:n
                if isDominant(M, row, column)
                    foundRow = row
                    foundColumn = column
                    found = true
                    break
                end
            end
            if found
                break
            end
        end
        if foundRow != i
            temp = copy(M[i, :])
            M[i, :] = M[foundRow, :]
            M[foundRow, :] = temp
            right[i], right[foundRow] = right[foundRow], right[i]
        end
        if foundColumn != i
            temp = copy(M[:, i])
            M[:, i] = M[:, foundColumn]
            M[:, foundColumn] = temp
            order[i], order[foundColumn] = order[foundColumn], order[i]
        end
    end
    return M, right, order
end

function restoreCoordinates(x, order)
    result = zeros(Float64, length(x))
    for i in 1:length(x)
        result[order[i]] = x[i]
    end
    return result
end

function lapackSolve(A, b)
    Acopy = copy(A)
    bcopy = copy(b)
    x, factors, pivots = LAPACK.gesv!(Acopy, bcopy)
    return x
end

function percentError(x, x_t)
    return norm(x - x_t, 2) / norm(x_t, 2) * 100
end

A = generateMatrix(n, k)
x_t = ones(Float64, n)
b = A * x_t

xGauss = gaussSolve(A, b)
errorGauss = percentError(xGauss, x_t)

Arows, brows = reorderRows(A, b)
xRows = gaussSolve(Arows, brows)
errorRows = percentError(xRows, x_t)

Acolumns, columnOrder = reorderColumns(A)
xColumnsChanged = gaussSolve(Acolumns, b)
xColumns = restoreCoordinates(xColumnsChanged, columnOrder)
errorColumns = percentError(xColumns, x_t)

Acombined, bcombined, combinedOrder = reorderCombined(A, b)
xCombinedChanged = gaussSolve(Acombined, bcombined)
xCombined = restoreCoordinates(xCombinedChanged, combinedOrder)
errorCombined = percentError(xCombined, x_t)

xLapack = lapackSolve(A, b)
errorLapack = percentError(xLapack, x_t)

println("n = ", n)
@printf("k = %.10f\n", k)
@printf("Метод Гаусса: %.15f %%\n", errorGauss)
@printf("После перестановки строк: %.15f %%\n", errorRows)
@printf("После перестановки столбцов: %.15f %%\n", errorColumns)
@printf("После перестановки строк и столбцов: %.15f %%\n", errorCombined)
@printf("LAPACK: %.15f %%\n", errorLapack)

n = 1000
k = 0.0100000000
Метод Гаусса: 0.000000000000228 %
После перестановки строк: 0.000000000000228 %
После перестановки столбцов: 0.000000000000228 %
После перестановки строк и столбцов: 0.000000000000228 %
LAPACK: 0.000000000000111 %
